# RQ6 – Robustness and Generalization

Kaggle-ready notebook. It loads the raw F1 strategy dataset, generates the actual table and publication-ready PDF figure for this research question, and saves results in `outputs/`.

In [ ]:

# Kaggle-ready setup: imports, dataset loading, preprocessing helpers
import os, glob, warnings, json
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.inspection import permutation_importance

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

RANDOM_STATE = 42
OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Publication plotting defaults
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'legend.fontsize': 9,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

def find_dataset_file():
    """Finds a CSV/XLSX file in Kaggle input or current working directory."""
    patterns = [
        '/kaggle/input/**/*.csv', '/kaggle/input/**/*.xlsx', '/kaggle/input/**/*.xls',
        './**/*.csv', './**/*.xlsx', './**/*.xls',
        '/mnt/data/*.csv', '/mnt/data/*.xlsx', '/mnt/data/*.xls'
    ]
    files = []
    for p in patterns:
        files.extend(glob.glob(p, recursive=True))
    files = [f for f in files if not os.path.basename(f).startswith('RQ')]
    if not files:
        raise FileNotFoundError('No CSV/XLSX dataset found. On Kaggle, attach the F1 Strategy Dataset using Add Data.')
    # Prefer f1 strategy file names, otherwise largest file
    preferred = [f for f in files if 'f1' in os.path.basename(f).lower() or 'strategy' in os.path.basename(f).lower()]
    candidates = preferred or files
    candidates = sorted(candidates, key=lambda f: os.path.getsize(f), reverse=True)
    return candidates[0]

def load_data():
    path = find_dataset_file()
    print('Loading dataset:', path)
    if path.lower().endswith('.csv'):
        df = pd.read_csv(path)
    else:
        df = pd.read_excel(path)
    print('Shape:', df.shape)
    print('Columns:', list(df.columns))
    return df

def validate_columns(df):
    required = {'PitNextLap'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'Missing required target column(s): {missing}')
    return True

def get_feature_columns(df, requested=None):
    leakage_cols = {'PitNextLap', 'PitStop'}
    if requested is None:
        cols = [c for c in df.columns if c not in leakage_cols]
    else:
        cols = [c for c in requested if c in df.columns and c not in leakage_cols]
    return cols

def make_preprocessor(X):
    cat_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
    num_cols = [c for c in X.columns if c not in cat_cols]
    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    return ColumnTransformer([
        ('num', numeric_pipeline, num_cols),
        ('cat', categorical_pipeline, cat_cols)
    ]), num_cols, cat_cols

def make_pipeline(model, X):
    preprocessor, _, _ = make_preprocessor(X)
    return Pipeline([('preprocess', preprocessor), ('model', model)])

def train_test(df, features=None, test_size=0.2):
    validate_columns(df)
    features = get_feature_columns(df, features)
    X = df[features].copy()
    y = df['PitNextLap'].astype(int).copy()
    return train_test_split(X, y, test_size=test_size, stratify=y, random_state=RANDOM_STATE)

def predict_scores(pipe, X_test):
    y_pred = pipe.predict(X_test)
    y_score = None
    if hasattr(pipe, 'predict_proba'):
        try:
            y_score = pipe.predict_proba(X_test)[:, 1]
        except Exception:
            y_score = None
    if y_score is None and hasattr(pipe, 'decision_function'):
        try:
            y_score = pipe.decision_function(X_test)
        except Exception:
            y_score = None
    return y_pred, y_score

def metric_row(model_name, y_true, y_pred, y_score=None):
    row = {
        'Model': model_name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1-score': f1_score(y_true, y_pred, zero_division=0),
    }
    if y_score is not None:
        try:
            row['AUC'] = roc_auc_score(y_true, y_score)
        except Exception:
            row['AUC'] = np.nan
    return row

def xgb_model():
    if HAS_XGB:
        return XGBClassifier(
            n_estimators=250, max_depth=4, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9,
            eval_metric='logloss', random_state=RANDOM_STATE,
            n_jobs=-1
        )
    return RandomForestClassifier(n_estimators=250, random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced')

def save_table(df, filename):
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(path, index=False)
    print('Saved table:', path)
    return path

def save_fig(filename):
    path = os.path.join(OUTPUT_DIR, filename)
    plt.tight_layout()
    plt.savefig(path, bbox_inches='tight')
    print('Saved figure:', path)
    plt.show()
    return path

def format_scores(df, cols=None):
    out = df.copy()
    cols = cols or [c for c in out.columns if c not in ['Model', 'Scenario', 'Feature Set', 'Feature', 'Rank', 'Criterion', 'Selected']]
    for c in cols:
        if c in out.columns and pd.api.types.is_numeric_dtype(out[c]):
            out[c] = out[c].round(4)
    return out


## RQ6: How robust is the selected model across validation and perturbation scenarios?

In [ ]:

df = load_data(); validate_columns(df)
features = get_feature_columns(df)
X = df[features].copy(); y = df['PitNextLap'].astype(int)
rows=[]

# Scenario 1: standard stratified split
X_train, X_test, y_train, y_test = train_test(df, features)
pipe = make_pipeline(xgb_model(), X_train)
pipe.fit(X_train, y_train)
y_pred, y_score = predict_scores(pipe, X_test)
row = metric_row('Standard Split', y_test, y_pred, y_score); row['Scenario']=row.pop('Model'); rows.append(row)

# Scenario 2: 5-fold cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
pipe_cv = make_pipeline(xgb_model(), X)
scoring = {'Accuracy':'accuracy', 'Precision':'precision', 'Recall':'recall', 'F1-score':'f1', 'AUC':'roc_auc'}
cv_scores = cross_validate(pipe_cv, X, y, cv=cv, scoring=scoring, n_jobs=-1)
row = {'Scenario':'5-Fold CV'}
for metric in scoring:
    vals = cv_scores[f'test_{metric}']
    row[metric] = vals.mean(); row[f'{metric} Std. Dev.'] = vals.std()
rows.append(row)

# Scenario 3: cross-race testing when Race exists
if 'Race' in df.columns:
    groups = df['Race']
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
    train_idx, test_idx = next(splitter.split(X, y, groups))
    Xtr, Xte = X.iloc[train_idx], X.iloc[test_idx]
    ytr, yte = y.iloc[train_idx], y.iloc[test_idx]
    pipe = make_pipeline(xgb_model(), Xtr); pipe.fit(Xtr, ytr)
    y_pred, y_score = predict_scores(pipe, Xte)
    row=metric_row('Cross-Race Testing', yte, y_pred, y_score); row['Scenario']=row.pop('Model'); rows.append(row)

# Scenario 4: noise added to numerical test features
X_train, X_test, y_train, y_test = train_test(df, features)
pipe = make_pipeline(xgb_model(), X_train); pipe.fit(X_train, y_train)
X_noisy = X_test.copy()
num_cols = X_noisy.select_dtypes(include=[np.number]).columns
rng = np.random.default_rng(RANDOM_STATE)
for c in num_cols:
    std = X_noisy[c].std()
    if pd.notna(std) and std > 0:
        X_noisy[c] = X_noisy[c] + rng.normal(0, 0.10 * std, size=len(X_noisy))
y_pred, y_score = predict_scores(pipe, X_noisy)
row=metric_row('10% Noise Added', y_test, y_pred, y_score); row['Scenario']=row.pop('Model'); rows.append(row)

results = format_scores(pd.DataFrame(rows))
save_table(results, 'RQ6_robustness_analysis.csv')
results


In [ ]:

plot_cols = ['Accuracy','Precision','Recall','F1-score']
plot_df = results.set_index('Scenario')[plot_cols]
ax = plot_df.plot(kind='bar', figsize=(9, 5), rot=15, edgecolor='black')
ax.set_title('RQ6. Robustness of Selected Model Across Validation Scenarios')
ax.set_xlabel('Scenario')
ax.set_ylabel('Score')
ax.set_ylim(0, 1)
ax.legend(ncol=4, loc='lower center', bbox_to_anchor=(0.5, -0.35), frameon=False)
save_fig('RQ6_robustness_analysis.pdf')
